# OpenComic 漫画批量超分 · 2×
首次连接请选择 **L4 GPU**，运行 **①**。环境就绪后，在 **②** 选择输入并运行。
- **上传漫画**　两个文本框都留空，运行后选择一个 ZIP 或 CBZ。
- **拷贝漫画**　将一话的网页地址粘贴到「章节链接」。显卡所在的云端会下载图片，按原页序处理。
- **大文件**　也可以先通过左侧「文件」上传，把运行时文件路径填入「已有压缩包路径」。
- **下载**　完成后浏览器会下载 CBZ。每页输出 PNG。上传模式保留图片的子目录、顺序和 ComicInfo.xml。同名成品分别保存。若浏览器拦住自动下载，可以从左侧 `/content/OpenComic-results` 下载。
- **下一本**　只需重跑 **②**。云端文件是临时的，请在删除运行时之前取回成品。下载完成后可在「代码执行程序」中断开并删除运行时，释放算力。

默认配置为 Quality V3 去伪影、Fast V4 Preview 去网点、Balanced V3 放大 2×，默认切块边长为 1024 像素，可在②或④调整。较大的切块通常更占显存，显存不足时可选较小值。TTA 默认关闭，保留大网点默认开启。章节解析参照 [ComicReadScript](https://github.com/dahaox3/ComicReadScript/tree/chainner-upscale/src/userscript/copyApi)，图像处理使用[官方 OpenComic 运行库](https://github.com/ollm/opencomic-ai-bin)和[已固定版本的适配代码](https://github.com/dahaox3/chaiNNer/tree/e2b3015c3e5b67ce14a021cd21a56bedd6dbacfb/backend/src/nodes/impl/opencomic)。
批量选读请运行 ①、③、④。运行③后，用下拉框选择分组、起始话和结束话，确认范围再运行④。解压下载的 ZIP 后，在 OpenComic 中打开 CBZ，章末可阅读拷贝漫画吐槽。每话保存在当前运行时，断开并销毁运行时后需要重新处理。


In [ ]:
#@title ① 准备 OpenComic 和显卡（每次连接运行一次）
import gzip
import hashlib
import json
import os
import re
import shutil
import subprocess
import tempfile
import zipfile
from pathlib import Path, PurePosixPath
from time import perf_counter
from urllib.parse import urlsplit

import requests
from PIL import Image, ImageOps
from google.colab import files
from tqdm.auto import tqdm

ROOT = Path('/content/opencomic-batch')
ROOT.mkdir(exist_ok=True)
RESULTS = Path('/content/OpenComic-results')
RESULTS.mkdir(exist_ok=True)
MODELS = ROOT / 'models'
NODE = ROOT / 'node_modules/node/bin/node'
SOURCE = 'https://raw.githubusercontent.com/dahaox3/chaiNNer/e2b3015c3e5b67ce14a021cd21a56bedd6dbacfb/backend/src/nodes/impl/opencomic/'

def fetch(url, **kwargs):
    response = requests.get(url, timeout=(12, 90), **kwargs)
    response.raise_for_status()
    return response

def run(args):
    p = subprocess.run(args, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=420)
    if p.returncode:
        raise RuntimeError(p.stdout[-2200:])
    return p.stdout

def prepare_runtime():
    if not shutil.which('nvidia-smi'):
        raise RuntimeError('请在「代码执行程序 → 更改运行时类型」中选择 L4 GPU，再运行第①步。')
    if not shutil.which('vulkaninfo'):
        run(['apt-get','update','-qq'])
        run(['apt-get','install','-y','-qq','vulkan-tools'])
    if not Path('/usr/share/vulkan/icd.d/nvidia_icd.json').exists():
        driver = run(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader']).strip().splitlines()[0]
        release = dict(s.split('=', 1) for s in Path('/etc/os-release').read_text().splitlines() if '=' in s)
        distro = release['VERSION_ID'].strip('"').replace('.','')
        base = f'https://developer.download.nvidia.com/compute/cuda/repos/ubuntu{distro}/x86_64/'
        index = gzip.decompress(fetch(base+'Packages.gz').content).decode()
        for block in index.split('\n\n'):
            fields = dict(l.split(': ', 1) for l in block.splitlines() if ': ' in l and not l.startswith(' '))
            if fields.get('Package') != 'libnvidia-gl-'+driver.split('.')[0] or fields.get('Architecture') != 'amd64' or fields.get('Version','').split('-')[0] != driver:
                continue
            payload = fetch(base+fields['Filename'].removeprefix('./')).content
            if hashlib.sha256(payload).hexdigest() != fields['SHA256']:
                raise ValueError('NVIDIA 下载校验失败')
            deb = ROOT/'nvidia-gl.deb'
            deb.write_bytes(payload)
            run(['dpkg-deb','-x',str(deb),'/'])
            run(['ldconfig'])
            deb.unlink()
            break
        else:
            raise RuntimeError('找不到与当前 GPU 驱动匹配的 Vulkan 组件：'+driver)
    info = run(['vulkaninfo','--summary'])
    devices = [l.split('=',1)[-1].strip() for l in info.splitlines() if 'deviceName' in l]
    if not any(re.search('nvidia|tesla', s, re.I) for s in devices):
        raise RuntimeError('没有识别到 NVIDIA Vulkan 设备')
    print('显卡：', ', '.join(devices), flush=True)
    for name in ['worker.cjs','patch-daemon-tta.cjs','catalog.json']:
        (ROOT/name).write_bytes(fetch(SOURCE+name).content)
    run(['npm','install','--no-audit','--no-fund','node@22','opencomic-ai-bin@1.6.1','sharp@0.35.5'])
    run([str(NODE),'patch-daemon-tta.cjs',str(ROOT)])
    return json.loads((ROOT/'catalog.json').read_text())

catalog = prepare_runtime()
print('环境已就绪。')

from contextlib import contextmanager, nullcontext
from html import unescape
from uuid import uuid4
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.padding import PKCS7

def chapter_pages(url):
    u = urlsplit(url.strip())
    match = re.fullmatch(r'/(?:comic/([^/]+)/chapter/([^/]+)|h5/comicContent/([^/]+)/([^/]+))/?', u.path)
    if u.scheme != 'https' or u.hostname not in {'mangacopy.com','www.mangacopy.com','copymanga.site','www.copymanga.site','copymanga.tv','www.copymanga.tv'} or not match:
        raise ValueError('请粘贴拷贝漫画的单话 HTTPS 地址。')
    comic, chapter = match.groups()[:2] if match[1] else match.groups()[2:]
    page = f'https://{u.hostname}/comic/{comic}/chapter/{chapter}'
    headers = {'User-Agent':'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/131.0.0.0 Safari/537.36'}
    html = fetch(page, headers=headers).text
    soup = BeautifulSoup(html, 'html.parser')
    title = soup.title.get_text(strip=True) if soup.title else chapter
    ordered = []
    # Follow ComicReadScript's HTML-first ordering and its c1500x WebP selection.
    def variable(name):
        m = re.search(r'\b(?:var|let|const)\s+' + name + r"""\s*=\s*['"]([^'"]+)['"]""", html)
        return m[1] if m else None
    raw = variable('contentKey')
    key = variable('cct') or variable('ccz')
    if raw and key:
        decrypt = Cipher(algorithms.AES(key.encode()), modes.CBC(raw[:16].encode())).decryptor()
        plain = decrypt.update(bytes.fromhex(raw[16:])) + decrypt.finalize()
        pad = PKCS7(128).unpadder()
        ordered = [item['url'] for item in json.loads(pad.update(plain) + pad.finalize())]
    if not ordered:
        ordered = [img['data-src'] for img in soup.select('.comicContent-list img[data-src]')]
    if not ordered:
        # The public chapter2 response uses a separate ordering array.
        for suffix in [f'chapter2/{chapter}', f'chapter/{chapter}?platform=3']:
            response = requests.get(f'https://{u.hostname}/api/v3/comic/{comic}/{suffix}', headers={**headers,'Referer':page,'Accept':'application/json'},timeout=20)
            if response.status_code in (401,403): response.raise_for_status()
            if not response.ok or 'json' not in response.headers.get('content-type',''): continue
            data = (response.json().get('results') or {}).get('chapter') or {}
            items, words = data.get('contents') or [], data.get('words') or []
            ordered = [items[i]['url'] for i in sorted(range(len(items)),key=lambda i: words[i] if i<len(words) else i)]
            if ordered: break
    if not ordered:
        raise ValueError('未取得本话图片。请确认链接可公开阅读，或改用压缩包。')
    urls = [urljoin(page, re.sub(r'c\d+x\.\w+', 'c1500x.webp', unescape(x))) for x in ordered]
    if not all(urlsplit(x).scheme == 'https' for x in urls):
        raise ValueError('章节包含异常的图片地址')
    return title, urls, {**headers,'Referer':page}

def prepare_models(steps):
    def names(s):
        yield s['model']
        for child in ('keepBigHalftone','artifactRemoval'):
            if child in s: yield from names(s[child])
    chosen = {name for s in steps for name in names(s)}
    paths = {f for group in catalog['models'].values() for name, meta in group.items() if name in chosen for f in meta['files']}
    for f in sorted(paths):
        path = MODELS/f
        size, sha = catalog['files'][f]
        def valid(b):
            return len(b)==size and hashlib.sha1(f'blob {size}\0'.encode()+b).hexdigest()==sha
        if path.is_file() and valid(path.read_bytes()): continue
        b = fetch(f"https://raw.githubusercontent.com/ollm/opencomic-ai-models/{catalog['revision']}/models/{f}").content
        if not valid(b): raise ValueError('模型校验失败：'+f)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(b)

@contextmanager
def worker(steps):
    p = None
    def close():
        nonlocal p
        if p is None: return
        try:
            try: p.stdin.close()
            except BrokenPipeError: pass
            try: p.wait(timeout=8)
            except subprocess.TimeoutExpired: p.kill(); p.wait()
        finally:
            p.stdout.close()
            p = None

    def process(source, dest):
        nonlocal p
        if p is not None and p.poll() is not None: close()
        if p is None:
            prepare_models(steps)
            p = subprocess.Popen([str(NODE),'worker.cjs',str(ROOT),str(MODELS)],cwd=ROOT,
                                 stdin=subprocess.PIPE,stdout=subprocess.PIPE,text=True)
        try:
            p.stdin.write(json.dumps(dict(source=str(source),dest=str(dest),steps=steps))+'\n')
            p.stdin.flush()
            for line in p.stdout:
                reply = json.loads(line)
                if reply.get('error'): raise RuntimeError(reply['error'])
                if reply.get('done'): return
        except BrokenPipeError:
            pass
        close()
        raise RuntimeError('OpenComic 已退出')

    try: yield process
    finally: close()

def convert(stream, temp, process):
    source, dest = temp/'page.png', temp/'processed.png'
    with Image.open(stream) as original:
        im = ImageOps.exif_transpose(original)
        alpha = im.convert('RGBA').getchannel('A') if 'A' in im.getbands() or 'transparency' in im.info else None
        im.convert('RGB').save(source)
        size = im.size
    dest.unlink(missing_ok=True)
    process(source,dest)
    with Image.open(dest) as result:
        if result.size != (size[0]*2,size[1]*2): raise ValueError('输出尺寸异常')
        if alpha is not None:
            combined = result.convert('RGB')
            combined.putalpha(alpha.resize(result.size, Image.Resampling.BILINEAR))
    if alpha is not None: combined.save(dest)
    return dest

def archive_entries(z):
    images = {'.png','.jpg','.jpeg','.webp','.avif','.bmp','.tif','.tiff','.gif'}
    entries, used = [], set()
    for member in z.infolist():
        path = PurePosixPath(member.filename.replace('\\','/'))
        if member.is_dir() or '__MACOSX' in path.parts or path.name.startswith('._'): continue
        if path.is_absolute() or '..' in path.parts or ':' in path.parts[0]: raise ValueError('压缩包包含无效路径')
        pic = path.suffix.lower() in images
        if not pic and path.name.lower() != 'comicinfo.xml': continue
        name = path.with_suffix('.png') if pic else path
        while str(name).casefold() in used: name = name.with_name(name.stem+'_'+name.suffix)
        used.add(str(name).casefold())
        entries.append((member,str(name),pic))
    if not any(pic for _,_,pic in entries): raise ValueError('压缩包中没有可处理的漫画图片')
    return entries

def run_batch(link, path, steps, auto_download, destination=None, process=None):
    import io
    with tempfile.TemporaryDirectory(dir=ROOT) as scratch:
        temp = Path(scratch)
        if link:
            title, urls, headers = chapter_pages(link)
            print(f'已找到 {len(urls)} 页：{title}', flush=True)
        else:
            if not path:
                uploaded = files.upload(target_dir=str(temp))
                if len(uploaded)!=1: raise ValueError('每次请选择一个 ZIP 或 CBZ 文件')
                path = next(p for p in temp.iterdir() if p.is_file())
                del uploaded
            path = Path(path)
            if not zipfile.is_zipfile(path): raise ValueError('请选择有效的 ZIP 或 CBZ 文件')
            title = path.stem
        clean = re.sub(r'[\\/:*?"<>|\x00-\x1f]', '_',title).strip(' .')[:90] or '漫画'
        result = destination or RESULTS/(clean+'-OpenComic-2x-'+uuid4().hex[:8]+'.cbz')
        result.parent.mkdir(parents=True, exist_ok=True)
        partial = result.with_suffix('.part')
        try:
            session = worker(steps) if process is None else nullcontext(process)
            with session as upscale, zipfile.ZipFile(partial,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as output:
                if link:
                    output.comment = chapter_source(link)
                    for i,url in enumerate(tqdm(urls,desc='云端下载并超分',unit='页'),1):
                        page = fetch(url,headers=headers)
                        output.write(convert(io.BytesIO(page.content),temp,upscale),f'{i:04}.png')
                else:
                    with zipfile.ZipFile(path) as source:
                        output.comment = chapter_source(source.comment.decode('utf-8', errors='replace'))
                        for member,name,pic in tqdm(archive_entries(source),desc='正在超分',unit='项'):
                            try:
                                with source.open(member) as stream:
                                    if pic: output.write(convert(stream,temp,upscale),name)
                                    else: output.writestr(name,stream.read())
                            except Exception as e: raise RuntimeError(f'处理失败：{member.filename}') from e
            partial.rename(result)
        finally:
            partial.unlink(missing_ok=True)
    print(f'完成：{result}（{result.stat().st_size/1024**2:.1f} MiB）',flush=True)
    if auto_download: files.download(str(result))
    return result

def chapter_source(value):
    try:
        u = urlsplit(value.strip())
        if u.scheme != 'https' or u.hostname not in COPY_HOSTS:
            return b''
        match = re.fullmatch(r'/(?:comic/([\w-]+)/chapter/([\w-]+)|h5/comicContent/([\w-]+)/([\w-]+))/?', u.path)
        if not match:
            return b''
        comic, chapter = match.groups()[:2] if match[1] else match.groups()[2:]
        return f'https://www.mangacopy.com/comic/{comic}/chapter/{chapter}'.encode()
    except ValueError:
        return b''

COPY_HOSTS = {'mangacopy.com','www.mangacopy.com','copymanga.site','www.copymanga.site','copymanga.tv','www.copymanga.tv'}
def catalog_api(path, params=None):
    headers = {'User-Agent':'Mozilla/5.0', 'x-requested-with':'com.manga2020.app', 'platform':'3',
               'version':'2024.4.28','webp':'1','accept':'application/json','Referer':'https://www.mangacopy.com/'}
    data = fetch('https://api.2024manga.com/api/v3/'+path, headers=headers, params=params).json()
    if data.get('code') != 200:
        raise RuntimeError(data.get('message', '目录请求失败'))
    return data['results']

def load_book(link):
    u = urlsplit(link.strip())
    m = re.fullmatch(r'/(?:h5/details/)?comic/([\w-]+)/?', u.path)
    if u.scheme != 'https' or u.hostname not in COPY_HOSTS or not m:
        raise ValueError('请粘贴拷贝漫画作品目录 HTTPS 链接')
    slug = m[1]
    data = catalog_api('comic2/'+slug)
    groups = data.get('groups') or [{'name':'默认','path_word':'default'}]
    if isinstance(groups, dict):
        groups = list(groups.values())
    return slug, data['comic']['name'], groups

def chapter_list(slug, group):
    rows = []
    while True:
        page = catalog_api(f'comic/{slug}/group/{group}/chapters', {'limit':100, 'offset':len(rows), '_update':'true'})
        rows.extend(page['list'])
        if len(rows) >= page['total']:
            return rows
        if not page['list']:
            raise RuntimeError('目录分页不完整，请重试')

def safe_name(value):
    return re.sub(r'[\\/:*?"<>|\x00-\x1f]', '_', value).strip(' .')[:80] or '漫画'

def process_range(slug, title, group, rows, start, end, steps, auto_download=True):
    if not (1 <= start <= end <= len(rows)):
        raise ValueError('起止序号超出目录范围')
    variant = hashlib.sha256(json.dumps(steps, sort_keys=True).encode()).hexdigest()[:16]
    if not re.fullmatch(r'[\w-]+', slug) or not re.fullmatch(r'[\w-]+', group):
        raise ValueError('作品或分组编号无效')
    folder = RESULTS/'batch'/slug/group/variant
    folder.mkdir(parents=True, exist_ok=True)
    products, failures = [], []
    with worker(steps) as process:
        for position in range(start, end+1):
            ch = rows[position-1]
            chapter_id = ch['uuid']
            if not re.fullmatch(r'[a-zA-Z0-9-]+', chapter_id):
                raise ValueError('章节 ID 无效')
            label = f"{position:04} - {safe_name(ch['name'])}"
            chapter_folder = folder/chapter_id
            dest = next(chapter_folder.glob('*.cbz'), chapter_folder/(label+'.cbz'))
            link = f'https://www.mangacopy.com/comic/{slug}/chapter/{chapter_id}'
            try:
                if not dest.is_file():
                    run_batch(link, '', steps, False, destination=dest, process=process)
                else:
                    print('已完成，跳过：'+label)
                products.append((dest, label+'.cbz'))
            except Exception as exc:
                failures.append(label)
                print(f'{label} 失败：{exc}', flush=True)
    if not products:
        raise RuntimeError('本批没有完成的章节。重新运行本格可重试。')
    bundle = RESULTS/f'{safe_name(title)}-{start:04}-{end:04}-{uuid4().hex[:8]}.zip'
    with zipfile.ZipFile(bundle.with_suffix('.part'), 'w', allowZip64=True) as z:
        for product, label in products:
            z.write(product, f'{safe_name(title)}/{label}')
    bundle.with_suffix('.part').replace(bundle)
    print(f'已打包 {len(products)} 话：{bundle}')
    if failures:
        print('重跑同一范围，仅重试未完成章节：'+ '、'.join(failures))
    if auto_download:
        files.download(str(bundle))
    return bundle

def processing_steps(开启TTA=False, 保留大网点=True, tile_size=1024):
    options = dict(tileSize=int(tile_size), tta=开启TTA)
    descreen = dict(model='opencomic-ai-descreen-hard-fast-v4-preview', **options)
    if 保留大网点:
        descreen['keepBigHalftone'] = dict(model='opencomic-ai-descreen-mask-fast-v3-256',minSize=3, **options)
    return [
        dict(model='opencomic-ai-artifact-removal-quality-v3-ps', **options),
        descreen,
        dict(model='opencomic-ai-upscale-balanced-v3',scale=2,noise=0, **options),
    ]


In [ ]:
#@title ② 处理一话（修改后点击本格左侧 ▶，可以重复使用）
章节链接 = "https://www.mangacopy.com/comic/naoodngxueshenghui/chapter/b6f50cb6-65f9-11ee-bf39-55b00c27fb36" #@param {type:"string"}
已有压缩包路径 = "" #@param {type:"string"}
切块大小 = "1024" #@param ["256", "512", "1024", "1536", "2048"]
开启TTA = False #@param {type:"boolean"}
保留大网点 = True #@param {type:"boolean"}
自动下载成品 = True #@param {type:"boolean"}

if 章节链接.strip() and 已有压缩包路径.strip(): raise ValueError('请选择一种输入方式')
成品 = run_batch(章节链接.strip(), 已有压缩包路径.strip(), processing_steps(开启TTA, 保留大网点, 切块大小), 自动下载成品)


In [ ]:
#@title ③ 批量 · 加载作品目录
作品链接 = "https://www.mangacopy.com/comic/naoodngxueshenghui" #@param {type:"string"}
from html import escape
from uuid import uuid4
from IPython.display import HTML, JSON, display
from google.colab import output

batch_selection = None
slug, title, groups = load_book(作品链接)
picker_id = uuid4().hex
group_rows = {}

def select_range(picker, group_index, first=1, last=1, load=False):
    global batch_selection
    if picker != picker_id:
        raise ValueError('请使用最新运行的目录。')
    batch_selection = None
    if not 0 <= group_index < len(groups):
        raise ValueError('请选择有效分组。')
    group = groups[group_index]['path_word']
    if group not in group_rows:
        group_rows[group] = chapter_list(slug, group)
    rows = group_rows[group]
    if rows:
        if not 1 <= first <= last <= len(rows):
            raise ValueError('请选择有效的章节范围。')
        batch_selection = (slug, title, group, rows, first, last)
        summary = f'{title} · {rows[first-1]["name"]} 至 {rows[last-1]["name"]}，共 {last-first+1} 话'
    else:
        summary = '此分组暂无章节'
    result = {'summary': summary}
    if load:
        result['names'] = [row['name'] for row in rows]
    return result

def on_selection(*args):
    return JSON(select_range(*args))

output.register_callback('opencomic.select_range', on_selection)
initial = select_range(picker_id, 0, load=True)
def options(names, numbered=False):
    return ''.join(f'<option value="{i}">{str(i+1)+" · " if numbered else ""}{escape(name)}</option>'
                   for i, name in enumerate(names))

chapter_options = options(initial['names'], numbered=True)
controls = f'''
<div id="catalog-{picker_id}" style="display:grid;grid-template-columns:max-content minmax(0,18rem);gap:6px 10px;align-items:center;width:min(100%,24rem);font:14px system-ui">
  <label for="g-{picker_id}">分组</label><select id="g-{picker_id}">{options([g['name'] for g in groups])}</select>
  <label for="s-{picker_id}">起始话</label><select id="s-{picker_id}">{chapter_options}</select>
  <label for="e-{picker_id}">结束话</label><select id="e-{picker_id}">{chapter_options}</select>
  <span role="status" style="grid-column:1/-1">{escape(initial['summary'])}</span>
</div>'''
display(HTML(controls + """
<script>
(() => {
  const box = document.querySelector('[id^="catalog-"]');
  const id = box.id.slice(8);
  const [group, start, end] = box.querySelectorAll('select');
  const status = box.querySelector('[role="status"]');
  async function update(load) {
    group.disabled = start.disabled = end.disabled = true;
    status.textContent = load ? '正在读取章节…' : '正在确认范围…';
    try {
      const reply = await google.colab.kernel.invokeFunction('opencomic.select_range',
        [id, group.selectedIndex, load ? 1 : start.selectedIndex+1, load ? 1 : end.selectedIndex+1, load], {});
      const result = reply.data['application/json'];
      if (load) {
        for (const select of [start, end]) {
          select.replaceChildren(...result.names.map((name, i) => new Option((i+1)+' · '+name, i)));
        }
      }
      status.textContent = result.summary;
    } catch (error) {
      start.replaceChildren();
      end.replaceChildren();
      status.textContent = '读取选择失败，请重跑③：'+error.message;
    } finally {
      group.disabled = false;
      start.disabled = end.disabled = !start.length;
    }
  }
  group.onchange = () => update(true);
  start.onchange = () => {
    if (end.selectedIndex < start.selectedIndex) end.selectedIndex = start.selectedIndex;
    update(false);
  };
  end.onchange = () => {
    if (start.selectedIndex > end.selectedIndex) start.selectedIndex = end.selectedIndex;
    update(false);
  };
  start.disabled = end.disabled = !start.length;
})();
</script>"""))

In [ ]:
#@title ④ 批量 · 开始超分（重跑可重试失败章节）
批量切块大小 = "1024" #@param ["256", "512", "1024", "1536", "2048"]
批量TTA = False #@param {type:"boolean"}
批量保留大网点 = True #@param {type:"boolean"}
自动下载整批 = True #@param {type:"boolean"}
if not globals().get('batch_selection'):
    raise ValueError('请先运行第③格并确认选中的章节范围。')
批量成品 = process_range(*batch_selection, processing_steps(批量TTA, 批量保留大网点, 批量切块大小), 自动下载整批)

In [ ]:
#@title ⑤ 重新下载已完成的 CBZ 或整批 ZIP（留空下载最新一份）
文件名关键词 = "" #@param {type:"string"}
from pathlib import Path
import hashlib
from google.colab import files

def download_finished(keyword):
    folder = Path('/content/OpenComic-results')
    archives = sorted((p for p in folder.rglob('*') if p.suffix in {'.cbz', '.zip'}), key=lambda f: f.stat().st_mtime_ns, reverse=True)
    if not archives:
        print('当前运行时没有成品。刷新后请连接原运行时；如果原运行时已删除，请重新处理。')
        return
    print('可下载的成品（最新在前）')
    for archive in archives:
        print(f'{archive.relative_to(folder)}（{archive.stat().st_size / 1024**2:.1f} MiB）')
    keyword = keyword.strip().casefold()
    matches = [archive for archive in archives if keyword in str(archive.relative_to(folder)).casefold()] if keyword else archives[:1]
    if len(matches) != 1:
        print('请从列表复制唯一的文件名或末尾编号，填入「文件名关键词」后重跑本格。')
        return
    chosen = matches[0]
    with chosen.open('rb') as stream:
        digest = hashlib.file_digest(stream, 'sha256').hexdigest()
    print(f'正在下载：{chosen.name}\n字节数：{chosen.stat().st_size}\nSHA-256：{digest}', flush=True)
    files.download(str(chosen))

download_finished(文件名关键词)
